## 数据
- SebastianRaschka：《The Verdict》
- Andrew Karparthy

In [7]:
import urllib.request
import re

### 1. 下载The Verdict数据

In [3]:
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "data/the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('data/the-verdict.txt', <http.client.HTTPMessage at 0x10ba5cf80>)

In [14]:
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [17]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
processed = [item for item in result if item.strip()]
print(len(processed), processed[:100])

4690 ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']


### 2. Tokenizer:单词转换为token ID

In [18]:
all_words = sorted(set(processed))
vocab_size = len(all_words)
print(f"vocab size: {vocab_size}")

vocab size: 1130


In [22]:
vocab = {word:no for no, word in enumerate(all_words)}
for i, w in enumerate(vocab.items()):
    if i < 20:
        print(f"word to int: {w}")

word to int: ('!', 0)
word to int: ('"', 1)
word to int: ("'", 2)
word to int: ('(', 3)
word to int: (')', 4)
word to int: (',', 5)
word to int: ('--', 6)
word to int: ('.', 7)
word to int: (':', 8)
word to int: (';', 9)
word to int: ('?', 10)
word to int: ('A', 11)
word to int: ('Ah', 12)
word to int: ('Among', 13)
word to int: ('And', 14)
word to int: ('Are', 15)
word to int: ('Arrt', 16)
word to int: ('As', 17)
word to int: ('At', 18)
word to int: ('Be', 19)


In [29]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.word_to_int = vocab
        self.int_to_word = {i:w for w, i in vocab.items()}

    def encode(self, text):
        result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        processed = [item.strip() for item in result if item.strip()]
        return [self.word_to_int[item] for item in processed]

    def decode(self, ids):
        text = " ".join([self.int_to_word[i] for i in ids])
        text = re.sub(r'\s+([,.?"()\'!])', r'\1', text)
        return text
        

In [30]:
tokenizer = SimpleTokenizer(vocab)
text = """"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [31]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


### 3. 处理特殊词元
词汇表新增两个特殊词元：
- "\<unk\>"
- "\<end\>"

不同大模型会引入不同的特殊token，常用的如下：
- [BOS]: begin of sequence，标记文本的起点，告知大语言模型一段内容的开始。
- [EOS]: end of sequence，位于文本的末尾，类似<|endoftext|>，特别适用于连接多个不相关的文本。
- [PAD] : 填充, **当模型在批量输入上进行训练时，我们通常使用掩码技术，这意味着我们并不会关注那些仅用于填充的词元。因此，具体选择哪种词元来进行填充实际上并不重要**

In [69]:
class SimpleTokenizer:
    def __init__(self, text_file="data/the-verdict.txt"):
        with open(text_file, "r", encoding="utf-8") as f:
            raw_text = f.read()
        result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
        processed = [item.strip() for item in result if item.strip()]

        self.UNK_TOKEN = "<|unk|>"
        self.END_TOKEN = "<|end|>"

        self.all_tokens = sorted(list(set(processed)))
        self.all_tokens.extend([self.UNK_TOKEN, self.END_TOKEN])
        
        self.word_to_int = {w:i for i, w in enumerate(self.all_tokens)}
        self.int_to_word = {i:w for w, i in self.word_to_int.items()}

    def encode(self, text):
        result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        processed = [item.strip() for item in result if item.strip()]
        return [self.word_to_int[item] if item in self.all_tokens else self.word_to_int["<|unk|>"] for item in processed]

    def decode(self, ids):
        text = " ".join([self.int_to_word[i] for i in ids])
        text = re.sub(r'\s+([,.?"()\'!])', r'\1', text)
        return text

In [70]:
tokenizer = SimpleTokenizer()

In [71]:
tokenizer.word_to_int["<|unk|>"],tokenizer.word_to_int["<|end|>"], len(tokenizer.word_to_int),tokenizer.all_tokens[-5:]

(1130, 1131, 1132, ['younger', 'your', 'yourself', '<|unk|>', '<|end|>'])

In [72]:
text = """"It's the last he painted, you know, Jay Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1130, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know, <|unk|> Mrs. Gisburn said with pardonable pride.


### 4. BPE（Byte-Pair Encoding，字节对编码）
GPT-2、GPT-3都是使用的BPE分词器。\
现代大语言模型最主流的子词分词算法，核心是从字符出发，迭代合并高频相邻符号对，生成兼顾词表大小与未登录词（OOV）处理能力的混合词表。

#### 关键变体：字节级BPE（byte-level BPE）

In [77]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [79]:
tokenizer = tiktoken.get_encoding("gpt2")

In [89]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace. haveatry"
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13, 423, 265, 563]


In [90]:
tokenizer.decode(integers)

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace. haveatry'

In [91]:
print("vocab size: ", tokenizer.n_vocab)

vocab size:  50257


In [10]:
import tiktoken

encoding = tiktoken.get_encoding("gpt2")

text = "Hello, world! 你好，世界！GPT-2 分词测试。"

tokens = encoding.encode(text)
print("文本对应的 token 列表：", tokens)
# 输出示例：[15496, 11, 995, 0, 1252, 136, 245, 11040, 136, 245, 11041, 0, 30, 4183, 220, 10528, 21631, 13]

decoded_text = encoding.decode(tokens)
print("解码后的文本：", decoded_text)
# 输出：Hello, world! 你好，世界！GPT-2 分词测试。

# 5. 计算 token 数量（核心需求之一）ti
token_count = len(tokens)
print("文本的 token 数量：", token_count)
# 输出示例：18

# 6. 进阶：按字符拆分，查看每个字符/词组对应的 token（方便理解分词逻辑）
# for char in text:
#     char_tokens = encoding.encode(char)
#     print(f"字符「{char}」对应的 token：{char_tokens} (数量：{len(char_tokens)})")

文本对应的 token 列表： [15496, 11, 995, 0, 220, 19526, 254, 25001, 121, 171, 120, 234, 10310, 244, 45911, 234, 171, 120, 223, 38, 11571, 12, 17, 10263, 230, 228, 46237, 235, 38184, 233, 46237, 243, 16764]
解码后的文本： Hello, world! 你好，世界！GPT-2 分词测试。
文本的 token 数量： 33


In [5]:
encoding.decode([15496]), encoding.decode([11]), encoding.decode([995])

('Hello', ',', ' world')

In [11]:
print("\n\n理解token和文本的关系：")

# 7. 理解token和文本的关系
for ix, id in enumerate(tokens):
    print(f"===={ix}: token id: {id}, token text: {encoding.decode([id])}")




理解token和文本的关系：
====0: token id: 15496, token text: Hello
====1: token id: 11, token text: ,
====2: token id: 995, token text:  world
====3: token id: 0, token text: !
====4: token id: 220, token text:  
====5: token id: 19526, token text: �
====6: token id: 254, token text: �
====7: token id: 25001, token text: �
====8: token id: 121, token text: �
====9: token id: 171, token text: �
====10: token id: 120, token text: �
====11: token id: 234, token text: �
====12: token id: 10310, token text: �
====13: token id: 244, token text: �
====14: token id: 45911, token text: �
====15: token id: 234, token text: �
====16: token id: 171, token text: �
====17: token id: 120, token text: �
====18: token id: 223, token text: �
====19: token id: 38, token text: G
====20: token id: 11571, token text: PT
====21: token id: 12, token text: -
====22: token id: 17, token text: 2
====23: token id: 10263, token text:  �
====24: token id: 230, token text: �
====25: token id: 228, token text: �
====26: toke